# 네이버 블로그 크롤러 + Gemini AI 분석

특정 카테고리의 최근 글을 크롤링하고 Gemini AI로 투자 콘텐츠를 생성합니다.

## 기능
1. 특정 카테고리의 최근 N일 내 글 목록 가져오기
2. 각 글의 제목, 본문, 날짜 추출
3. 엑셀 파일로 저장
4. Gemini API로 투자 콘텐츠 생성

## 1. 패키지 설치

In [ ]:
!pip install -q google-colab-selenium beautifulsoup4 openpyxl google-generativeai

print("✅ 설치 완료!")

## 2. 라이브러리 임포트

In [ ]:
import time
import re
from datetime import datetime, timedelta
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import google.generativeai as genai
from google.colab import files

print("✅ 라이브러리 임포트 완료!")

## 3. 설정

In [ ]:
# 크롤링 설정
BLOG_ID = "yminsong"        # 블로그 ID
CATEGORY_NO = 31           # 카테고리 번호
DAYS = 14                  # 최근 며칠 이내 글

# Gemini API 키 (https://makersuite.google.com/app/apikey 에서 발급)
GEMINI_API_KEY = ""  # 여기에 API 키 입력

print(f"블로그 ID: {BLOG_ID}")
print(f"카테고리: {CATEGORY_NO}")
print(f"기간: 최근 {DAYS}일")
print(f"Gemini API: {'설정됨' if GEMINI_API_KEY else '미설정'}")

## 4. 함수 정의

In [ ]:
def setup_driver():
    """Chrome WebDriver 설정"""
    chrome_options = Options()
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--disable-infobars')
    driver = gs.Chrome(options=chrome_options)
    return driver


def extract_blog_title(html):
    """블로그 글의 타이틀 추출"""
    try:
        title_elem = html.select_one("div.se-module.se-module-text.se-title-text")
        if title_elem:
            return title_elem.get_text(strip=True)
        
        title_p = html.select_one("div.se-title-text p.se-text-paragraph")
        if title_p:
            return title_p.get_text(strip=True)
        
        return ""
    except:
        return ""


def extract_blog_content_only(html):
    """블로그 콘텐츠만 추출 (타이틀 제외)"""
    try:
        content_container = html.select("div.se-main-container")
        if not content_container:
            return ""
        
        content = ''.join(str(content_container))
        content = re.sub('<[^>]*>', '', content)
        content = content.replace('\n', ' ').replace('\u200b', '').replace('&ZeroWidthSpace;', '')
        content = re.sub(r'\s+', ' ', content).strip()
        
        return content
    except:
        return ""


def get_recent_posts_from_category(blog_id, category_no, days=14):
    """특정 카테고리의 최근 글 목록 가져오기"""
    driver = None
    posts = []
    
    try:
        driver = setup_driver()
        category_url = f"https://blog.naver.com/PostList.naver?blogId={blog_id}&categoryNo={category_no}"
        print(f"카테고리 페이지 접속: {category_url}")
        
        driver.get(category_url)
        time.sleep(3)
        
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "mainFrame"))
        )
        driver.switch_to.frame(iframe)
        time.sleep(2)
        
        html = BeautifulSoup(driver.page_source, "html.parser")
        table = html.select_one("table.blog2_list.blog2_categorylist")
        
        if not table:
            print("⚠️ 글 목록 테이블을 찾을 수 없습니다")
            return posts
        
        now = datetime.now()
        cutoff_date = now - timedelta(days=days)
        
        rows = table.select("tbody tr")
        print(f"총 {len(rows)}개의 글 발견")
        
        for row in rows:
            try:
                title_elem = row.select_one("td.title a")
                if not title_elem:
                    continue
                
                title = title_elem.get_text(strip=True)
                url = title_elem.get('href', '')
                
                date_elem = row.select_one("td.date span.date")
                if not date_elem:
                    continue
                
                date_str = date_elem.get_text(strip=True)
                
                try:
                    date_clean = date_str.replace(' ', '').replace('.', '-').rstrip('-')
                    post_date = datetime.strptime(date_clean, "%Y-%m-%d")
                    
                    if post_date >= cutoff_date:
                        posts.append({
                            'title': title,
                            'url': url if url.startswith('http') else f"https://blog.naver.com{url}",
                            'date': date_str,
                            'date_obj': post_date
                        })
                        print(f"✅ [{date_str}] {title}")
                    else:
                        print(f"⏭️  [{date_str}] {title} (기간 외)")
                
                except ValueError:
                    continue
            
            except Exception as e:
                continue
        
        print(f"\n✅ 최근 {days}일 이내 글: {len(posts)}개")
        return posts
    
    finally:
        if driver:
            driver.quit()


def extract_full_post(url):
    """개별 글의 전체 내용 추출"""
    driver = None
    try:
        driver = setup_driver()
        print(f"  글 접속 중: {url}")
        driver.get(url)
        time.sleep(3)
        
        iframe = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "mainFrame"))
        )
        driver.switch_to.frame(iframe)
        time.sleep(2)
        
        html = BeautifulSoup(driver.page_source, "html.parser")
        title = extract_blog_title(html)
        content = extract_blog_content_only(html)
        
        return {'title': title, 'content': content}
    
    except Exception as e:
        print(f"  ❌ 추출 실패: {e}")
        return {'title': '', 'content': ''}
    
    finally:
        if driver:
            driver.quit()

print("✅ 함수 정의 완료!")

## 5. 최근 글 목록 가져오기

In [ ]:
print("="*80)
print("1단계: 최근 글 목록 가져오기")
print("="*80)

recent_posts = get_recent_posts_from_category(BLOG_ID, CATEGORY_NO, DAYS)

if recent_posts:
    print(f"\n✅ {len(recent_posts)}개의 글을 찾았습니다!")
else:
    print("\n⚠️ 최근 글이 없습니다.")

## 6. 각 글의 전체 내용 추출

In [ ]:
if recent_posts:
    print("\n" + "="*80)
    print("2단계: 각 글의 전체 내용 추출")
    print("="*80)
    
    results = []
    
    for idx, post in enumerate(recent_posts, 1):
        print(f"\n[{idx}/{len(recent_posts)}] {post['title']}")
        
        full_post = extract_full_post(post['url'])
        
        results.append({
            'title': full_post['title'] or post['title'],
            'content': full_post['content'],
            'date': post['date'],
            'url': post['url']
        })
        
        print(f"  ✅ 완료 (본문 {len(full_post['content']):,}자)")
        
        if idx < len(recent_posts):
            time.sleep(2)
    
    print(f"\n✅ 총 {len(results)}개 글 추출 완료!")

## 7. 엑셀로 저장

In [ ]:
if results:
    print("\n" + "="*80)
    print("3단계: 엑셀 저장")
    print("="*80)
    
    df = pd.DataFrame(results)
    filename = 'naver_blog_posts.xlsx'
    
    df.to_excel(filename, index=False, engine='openpyxl')
    print(f"✅ 엑셀 파일 저장 완료: {filename}")
    print(f"   총 {len(results)}개 글 저장")
    
    # 파일 다운로드
    files.download(filename)
    print(f"✅ 파일 다운로드 시작: {filename}")
    
    # 데이터 미리보기
    print("\n데이터 미리보기:")
    display(df.head())

## 8. Gemini AI로 투자 콘텐츠 생성

In [ ]:
if results and GEMINI_API_KEY:
    print("\n" + "="*80)
    print("4단계: Gemini AI 분석")
    print("="*80)
    
    # Gemini 설정
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-pro')
    
    # 글 내용을 LLM이 이해하기 쉬운 형식으로 변환
    formatted_content = "# 블로그 글 모음\n\n"
    
    for idx, post in enumerate(results, 1):
        formatted_content += f"## 글 {idx}\n"
        formatted_content += f"**제목:** {post['title']}\n"
        formatted_content += f"**날짜:** {post['date']}\n"
        formatted_content += f"**내용:**\n{post['content'][:2000]}...\n"
        formatted_content += "\n" + "-"*80 + "\n\n"
    
    # 프롬프트
    prompt = f"""
# 지시문

첨부한 파일의 실제 토론 내용을 바탕으로 리테일 투자자(개미 투자자)들의 관심을 끌고, 장기적으로 유용한 지식을 제공할 수 있는 콘텐츠를 작성.

# 콘텐츠 구조: "미국 개미들의 투자노트"

## 1. 금주의 가장 뜨거운 질문 (The Big Question)
목표: 한 주간 커뮤니티에서 가장 논쟁적이었던 주제를 제시하여 독자의 흥미를 즉각적으로 유발합니다.

## 2. 이번 주 집중 분석 (Deep Dive of the Week)
목표: 특정 기업이나 산업에 대한 깊이 있는 분석을 공유합니다.

## 3. 타산지석: 실패에서 배우는 교훈 (The Cautionary Tale)
목표: 리테일 투자자들이 가장 공감하는 '실패 사례'를 통해 실질적인 교훈을 전달합니다.

## 4. 놓치면 안 될 시장의 신호들 (Market Radar)
목표: 큰 이슈는 아니지만, 미래의 투자 아이디어나 리스크가 될 수 있는 작은 신호들을 포착하여 공유합니다.

## 5. 핵심 요약 및 다음 주 관전 포인트 (Final Takeaway)
목표: 이번 주 콘텐츠의 핵심 교훈을 3~4줄로 요약하고, 다음 주에 주목해야 할 이벤트를 제시합니다.

# 분석할 블로그 글

{formatted_content}

위 블로그 글들을 바탕으로 "미국 개미들의 투자노트" 형식의 콘텐츠를 작성해주세요.
"""
    
    print("Gemini API 호출 중...")
    try:
        response = model.generate_content(prompt)
        ai_content = response.text
        
        print("\n" + "="*80)
        print("✅ Gemini AI 분석 완료!")
        print("="*80)
        
        # 결과 저장
        with open('gemini_analysis.txt', 'w', encoding='utf-8') as f:
            f.write(ai_content)
        
        files.download('gemini_analysis.txt')
        print("✅ 분석 결과 저장 및 다운로드 완료!")
        
        # 결과 출력
        print("\n" + "="*80)
        print("생성된 콘텐츠:")
        print("="*80)
        print(ai_content)
        
    except Exception as e:
        print(f"❌ Gemini API 에러: {e}")

elif not GEMINI_API_KEY:
    print("\n⚠️ Gemini API 키가 설정되지 않았습니다.")
    print("   API 키를 설정하면 AI 분석이 가능합니다.")
    print("   https://makersuite.google.com/app/apikey")

## 완료!

모든 작업이 완료되었습니다.

생성된 파일:
- `naver_blog_posts.xlsx` - 크롤링한 블로그 글 데이터
- `gemini_analysis.txt` - Gemini AI가 생성한 투자 콘텐츠